# Solution-treated (ST) reference comparison

    Fit the original ST scan and the reconstructed ST scan separately using the same reflections, peak model and lattice-parameter methods.

In [1]:
import json
from io import StringIO
from pathlib import Path

here = Path.cwd()
notebook_dir = next(
    path / "06_lattice_parameter_notebook"
    for path in (here, *here.parents)
    if (path / "06_lattice_parameter_notebook").is_dir()
)
batch_notebook = notebook_dir / "Lattice_parameter_construction_bulk.ipynb"
for cell in json.loads(batch_notebook.read_text(encoding="utf-8"))["cells"]:
    if cell["cell_type"] == "code" and "batch-run" not in cell["metadata"].get("tags", []):
        get_ipython().run_cell("".join(cell["source"]), silent=True).raise_error()

In [2]:
# Read the measured ST profile for the benchmark only.
data_block = reference_txt.read_text(encoding="utf-8-sig").split("[Data]", 1)[1]
raw = pd.read_csv(StringIO(data_block.strip()), usecols=[0, 1], header=0)
raw_peaks = fit_peaks(raw.iloc[:, 0].to_numpy(), raw.iloc[:, 1].to_numpy(), "ST original")
raw_used, raw_estimates = calculate_lattice(raw_peaks)

st_csv = input_dir / "XRD_ST_2015_reconstructed_v4.csv"
st = pd.read_csv(st_csv)
v4_peaks = fit_peaks(st["angle_degrees"].to_numpy(), st["intensity_estimated"].to_numpy(), "ST V4")
v4_used, v4_estimates = calculate_lattice(v4_peaks)

comparison = pd.DataFrame({"Original TXT (Å)": raw_estimates, "V4 CSV (Å)": v4_estimates})
comparison.index.name = "Method"
comparison["V4 minus original (Å)"] = comparison["V4 CSV (Å)"] - comparison["Original TXT (Å)"]
comparison["Absolute relative difference (%)"] = (
    comparison["V4 minus original (Å)"].abs() / comparison["Original TXT (Å)"] * 100
)
centres = pd.DataFrame({
    "Original 2θ (deg)": raw_used.set_index("hkl")["two_theta"],
    "V4 2θ (deg)": v4_used.set_index("hkl")["two_theta"],
})
centres["V4 minus original (deg)"] = centres["V4 2θ (deg)"] - centres["Original 2θ (deg)"]
output_dir.mkdir(exist_ok=True)
comparison.to_csv(output_dir / "st_reference_lattice_comparison.csv")
centres.to_csv(output_dir / "st_reference_peak_comparison.csv")
display(comparison.round(6), centres.round(6))

,Original TXT (Å),V4 CSV (Å),V4 minus original (Å),Absolute relative difference (%)
Method,,,,
Arithmetic mean (Å),4.054448,4.054361,-0.000087,0.002151
Nelson-Riley (Å),4.054048,4.053861,-0.000187,0.004614
Cohen LS + N-R (Å),4.053939,4.053963,0.000024,0.000585


,Original 2θ (deg),V4 2θ (deg),V4 minus original (deg)
hkl,,,
(200),44.662565,44.662404,-0.000161
(220),65.008774,65.013804,0.005030
(311),78.123198,78.123111,-0.000087
